# Biohub Cell Tracking During Development

Detects nuclei in each 3D timepoint with a scale-normalized Laplacian-of-Gaussian response, then links centroids across consecutive timepoints in physical micrometre coordinates using a constrained assignment that permits 1-to-2 parent/child matches for divisions. Outputs `submission.csv` in the required node/edge schema.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_laplace
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max


def _make_decompressor():
    try:
        from numcodecs import Blosc
        codec = Blosc()
        return lambda buf: codec.decode(buf)
    except Exception:
        pass
    try:
        import blosc as _b
        return lambda buf: _b.decompress(bytes(buf))
    except Exception:
        pass
    try:
        import blosc2 as _b2
        return lambda buf: _b2.decompress(bytes(buf))
    except Exception:
        pass
    try:
        import imagecodecs as _ic
        return lambda buf: _ic.blosc_decode(bytes(buf))
    except Exception:
        pass
    raise RuntimeError("no blosc decompressor available")


_decompress = _make_decompressor()

SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)
MAX_LINK_UM = 7.0
VOL_SHAPE = (64, 256, 256)
VOL_DTYPE = np.dtype("<u2")
DATASETS = ["44b6_0113de3b", "44b6_0b24845f", "6bba_05b6850b", "6bba_05db0fb1"]
TEST_ROOT = Path("/kaggle/input/biohub-cell-tracking-during-development/test")
OUT_PATH = Path("submission.csv")

In [ ]:
def load_timepoint(zarr_root, t):
    chunk_path = zarr_root / "0" / "c" / str(t) / "0" / "0" / "0"
    raw = chunk_path.read_bytes()
    decoded = _decompress(raw)
    arr = np.frombuffer(decoded, dtype=VOL_DTYPE)
    return arr.reshape(VOL_SHAPE)


def num_timepoints(zarr_root):
    chunk_dir = zarr_root / "0" / "c"
    return sum(1 for _ in chunk_dir.iterdir())

In [ ]:
def normalize(vol):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, (1.0, 99.5))
    if hi <= lo:
        return np.zeros_like(v)
    return np.clip((v - lo) / (hi - lo), 0.0, 1.0)


def detect(vol):
    v = normalize(vol)
    sigma_z = 6.0 / SCALE[0] / 2.355
    sigma_xy = 6.0 / SCALE[1] / 2.355
    response = -gaussian_laplace(v, sigma=(sigma_z, sigma_xy, sigma_xy))
    threshold = max(float(np.percentile(response, 99.5)), 0.005)
    min_dist_xy = max(int(round(4.0 / SCALE[1])), 2)
    coords = peak_local_max(
        response,
        min_distance=min_dist_xy,
        threshold_abs=threshold,
        exclude_border=False,
    )
    if coords.size == 0:
        return coords.reshape(0, 3)
    scores = response[coords[:, 0], coords[:, 1], coords[:, 2]]
    return coords[np.argsort(-scores)]

In [ ]:
def link_pair(prev_um, curr_um):
    if prev_um.shape[0] == 0 or curr_um.shape[0] == 0:
        return []
    d = np.linalg.norm(prev_um[:, None, :] - curr_um[None, :, :], axis=-1)
    feasible = d <= MAX_LINK_UM
    if not feasible.any():
        return []
    big = MAX_LINK_UM * 10.0
    base = np.where(feasible, d, big)
    cost = np.vstack([base, base])
    n = prev_um.shape[0]
    rows, cols = linear_sum_assignment(cost)
    seen = set()
    parent_children = {}
    for r, c in zip(rows, cols):
        if cost[r, c] >= big or int(c) in seen:
            continue
        parent = int(r % n)
        parent_children.setdefault(parent, []).append((float(cost[r, c]), int(c)))
        seen.add(int(c))
    edges = []
    for parent, kids in parent_children.items():
        kids.sort()
        if len(kids) == 1:
            edges.append((parent, kids[0][1]))
        else:
            d0, c0 = kids[0]
            d1, c1 = kids[1]
            if d1 <= 0.8 * MAX_LINK_UM and d1 - d0 <= 3.0:
                edges.append((parent, c0))
                edges.append((parent, c1))
            else:
                edges.append((parent, c0))
    return edges

In [ ]:
def track_dataset(zarr_root):
    T = num_timepoints(zarr_root)
    nodes = []
    edges = []
    next_id = 1
    prev_ids = []
    prev_um = np.zeros((0, 3), dtype=np.float64)
    for t in range(T):
        volume = load_timepoint(zarr_root, t)
        coords = detect(volume)
        ids = list(range(next_id, next_id + coords.shape[0]))
        next_id += coords.shape[0]
        for (z, y, x), nid in zip(coords, ids):
            nodes.append((nid, t, int(z), int(y), int(x)))
        curr_um = coords.astype(np.float64) * SCALE
        if t > 0 and prev_um.shape[0] and curr_um.shape[0]:
            for i, j in link_pair(prev_um, curr_um):
                edges.append((prev_ids[i], ids[j]))
        prev_ids = ids
        prev_um = curr_um
    return nodes, edges

In [ ]:
rows = []
gid = 0
for ds in DATASETS:
    nodes, edges = track_dataset(TEST_ROOT / f"{ds}.zarr")
    for nid, t, z, y, x in nodes:
        rows.append((gid, ds, "node", nid, t, z, y, x, -1, -1))
        gid += 1
    for s, d in edges:
        rows.append((gid, ds, "edge", -1, -1, -1, -1, -1, s, d))
        gid += 1

submission = pd.DataFrame(
    rows,
    columns=["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"],
)
submission.to_csv(OUT_PATH, index=False)
submission.head()